# 02 - Selección de Componentes: Scree Plot, Varianza Acumulada y Error de Reconstrucción

## Curso de Machine Learning: Modelos No Supervisados

El hiperparámetro central de PCA es el número de componentes principales a retener: **$k \in \{1, \dots, p\}$**.

- Si elegimos un $k$ **demasiado pequeño** (ej. $k=1$), subestimamos la dimensionalidad intrínseca y descartamos varianza real informativa.
- Si elegimos un $k$ **demasiado grande** (ej. $k=p$), no logramos ninguna reducción ni mitigación de ruido.

En este cuaderno estudiaremos las reglas matemáticas y criterios objetivos para determinar el número óptimo de componentes, la compresión de información y el **error cuadrático medio de reconstrucción ($MSE_{rec}$)**.

---

### Contenidos
1. Razón de Varianza Explicada Individual y Acumulada
2. Criterio 1: Gráfico de Sedimentación (*Scree Plot*) y Regla del Codo (Cattell, 1966)
3. Criterio 2: Umbral de Varianza Acumulada Mínima (80%, 90%, 95%) y Sintaxis Flotante en Scikit-Learn
4. Criterio 3: Criterio de Kaiser-Guttman ($\lambda_k \geq 1.0$ en Datos Estandarizados)
5. Matriz de Reconstrucción Inversa (`pca.inverse_transform`) y Error Cuadrático de Proyección
6. Demostración Práctica: Compresión y Reconstrucción Progresiva de Señales de Alta Dimensionalidad
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Razón de Varianza Explicada (*Explained Variance Ratio*)

Dado el conjunto de $p$ autovalores $\lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_p \geq 0$, la varianza total de la matriz de datos estandarizada es igual a la traza de la matriz de covarianza:
$$\text{Varianza Total} = \sum_{j=1}^{p} \text{Var}(x_j) = \sum_{j=1}^{p} \lambda_j = p \quad (\text{en datos estandarizados})$$

La **Razón de Varianza Explicada por la $k$-ésima componente** se define como:
$$\text{EVR}_k = \frac{\lambda_k}{\sum_{j=1}^{p} \lambda_j} \in [0, 1]$$

Y la **Varianza Explicada Acumulada** al retener las primeras $m$ componentes es:
$$\text{CEVR}_m = \sum_{k=1}^{m} \text{EVR}_k = \frac{\sum_{k=1}^{m} \lambda_k}{\sum_{j=1}^{p} \lambda_j}$$


In [ ]:
# Carga de datos de 13 variables fisicoquimicas (Wine)
wine = load_wine(as_frame=True)
X_scaled = StandardScaler().fit_transform(wine.data)

# Ajuste completo con p=13 componentes
pca_full = PCA().fit(X_scaled)
evr = pca_full.explained_variance_ratio_ * 100
cevr = np.cumsum(evr)
lambdas = pca_full.explained_variance_

df_varianzas = pd.DataFrame({
    'Componente': [f'PC{i+1}' for i in range(len(evr))],
    'Autovalor (lambda_k)': lambdas.round(4),
    'Varianza_Explicada (%)': evr.round(2),
    'Varianza_Acumulada (%)': cevr.round(2)
})
print("Desglose de Varianza por Componente Principal:")
display(df_varianzas)


## 2. Los Tres Criterios de Selección de $k$

### Criterio A: Gráfico de Sedimentación (*Scree Plot*) y Codo de Cattell
Se grafican los autovalores $\lambda_k$ frente a $k$. Se busca el punto de inflexión donde la curva pasa de una caída empinada (señal relevante) a una pendiente suave y plana (sedimento geológico / ruido residual).

### Criterio B: Criterio de Kaiser-Guttman ($\lambda_k \geq 1.0$)
Para datos estandarizados con `StandardScaler`, cada variable original tiene una varianza de $1.0$. Por tanto, **solo tiene sentido retener aquellas componentes que capturen al menos tanta información como una variable original individual** ($\lambda_k \geq 1.0$).

### Criterio C: Umbral de Varianza Acumulada Mínima (ej. $85\%$ o $90\%$)
Se retiene el menor número de componentes $m$ tal que $\text{CEVR}_m \geq \text{umbral}$. Scikit-Learn permite automatizar esto directamente pasando un número decimal flotante a `PCA(n_components=0.90)`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Panel 1: Scree Plot (Autovalores y Criterio de Kaiser)
componentes_idx = range(1, len(lambdas) + 1)
axes[0].plot(componentes_idx, lambdas, 'o-', color='#1f77b4', lw=2.2, label='Autovalor $\lambda_k$ (Varianza)')
axes[0].axhline(1.0, color='crimson', linestyle='--', lw=2, label='Regla de Kaiser ($\lambda = 1.0$)')
axes[0].axvline(3, color='darkorange', linestyle=':', lw=2, label='Codo Visual ($k=3$)')
axes[0].set_title('Gráfico de Sedimentación (Scree Plot)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Índice de Componente Principal', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Autovalor $\lambda_k$', fontsize=10, fontweight='bold')
axes[0].set_xticks(componentes_idx)
axes[0].legend(fontsize=10)

# Panel 2: Varianza Acumulada
axes[1].bar(componentes_idx, evr, color='#aec7e8', edgecolor='k', alpha=0.7, label='Varianza Individual (%)')
axes[1].plot(componentes_idx, cevr, 's-', color='#2ca02c', lw=2.2, label='Varianza Acumulada (%)')
axes[1].axhline(80.0, color='red', linestyle='--', lw=1.5, label='Umbral 80%')
axes[1].axhline(90.0, color='purple', linestyle='--', lw=1.5, label='Umbral 90%')
axes[1].set_title('Curva de Varianza Explicada Acumulada', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Número de Componentes Retenidas', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Porcentaje (%)', fontsize=10, fontweight='bold')
axes[1].set_xticks(componentes_idx)
axes[1].legend(fontsize=10)

plt.suptitle('Selección Multicriterio del Número de Componentes Principales', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Uso del paso flotante en Scikit-Learn
pca_90 = PCA(n_components=0.90).fit(X_scaled)
print(f"Componentes necesarias para retener al menos el 90% de varianza: {pca_90.n_components_} (de 13 originales)")


## 3. Reconstrucción Inversa y Error Cuadrático Medio ($MSE_{rec}$)

Dada la proyección $\mathbf{T}_k = \mathbf{X}_c \mathbf{V}_k$, podemos reconstruir una aproximación del dataset original en $\mathbb{R}^p$ mediante:
$$\hat{\mathbf{X}}_c = \mathbf{T}_k \mathbf{V}_k^T = \mathbf{X}_c \mathbf{V}_k \mathbf{V}_k^T$$

El **Error Cuadrático de Reconstrucción** equivale exactamente a la suma de los autovalores de las componentes que fueron descartadas:
$$MSE_{rec} = \frac{1}{n} \|\mathbf{X}_c - \hat{\mathbf{X}}_c\|_F^2 = \sum_{j=k+1}^{p} \lambda_j$$

Demostremos este teorema comprimiendo y reconstruyendo imágenes de dígitos manuscritos (8x8 = 64 dimensiones).


In [ ]:
digits = load_digits()
X_dig = digits.data

# Evaluemos el error de reconstruccion teorico vs practico al variar k de 1 a 64
pca_dig = PCA().fit(X_dig)
k_vals = [2, 8, 16, 32, 64]

fig, axes = plt.subplots(len(k_vals) + 1, 6, figsize=(12, 10))

# Imagenes originales (primeros 6 digitos)
for col in range(6):
    axes[0, col].imshow(X_dig[col].reshape(8, 8), cmap='binary')
    axes[0, col].axis('off')
axes[0, 0].set_title('Original (p=64)', fontsize=10, fontweight='bold', loc='left')

for row, k in enumerate(k_vals, start=1):
    pca_k = PCA(n_components=k).fit(X_dig)
    X_proj = pca_k.transform(X_dig)
    X_rec = pca_k.inverse_transform(X_proj)
    mse_rec = np.mean((X_dig - X_rec) ** 2)
    
    for col in range(6):
        axes[row, col].imshow(X_rec[col].reshape(8, 8), cmap='binary')
        axes[row, col].axis('off')
    axes[row, 0].set_title(f'k = {k} ({pca_k.explained_variance_ratio_.sum()*100:.0f}% var | MSE={mse_rec:.2f})',
                           fontsize=10, fontweight='bold', loc='left')

plt.suptitle('Compresión y Reconstrucción Progresiva con PCA (Dígitos 8x8)', fontsize=13, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


## 4. Conclusiones

1. **Scree Plot**: Visualiza la caída de autovalores; el punto de codo suele señalar el límite natural entre información estructural y ruido.
2. **Criterio de Kaiser**: Conserva aquellas componentes con $\lambda_k \geq 1.0$ (en variables estandarizadas).
3. **Umbral Flotante en Scikit-Learn**: Usar `PCA(n_components=0.95)` automatiza la selección del $k$ necesario para retener el 95% de variabilidad.
4. **Error de Reconstrucción**: Es inversamente proporcional a la varianza explicada acumulada; el residuo equivale a $\sum_{j=k+1}^p \lambda_j$.

En el siguiente cuaderno, **03_Interpretacion_Biplots_Cargas_y_Circulo_Correlaciones.ipynb**, aprenderemos a interpretar qué variables componen cada factor mediante el **Biplot** y el **Círculo de Correlaciones**.
